In [1]:
"""
FictiPay Churn Prediction Pipeline — v4
Observation window : 2024-01-01 → 2024-03-31  (strictly < 2024-04-01)
Prediction window  : 2024-04-01 → 2024-04-30  (not provided; defines label)
Evaluation metric  : AUC-ROC on float CHURN_PROB ∈ [0, 1]
"""

import os
import warnings
import numpy as np
import pandas as pd
import dask.dataframe as dd
from scipy import stats
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")

In [2]:
# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
DATA_DIR        = "/kaggle/input/competitions/bkash-presents-nsucec-datathon/public/"
TRX_DIR         = os.path.join(DATA_DIR, "transactions/")
BAL_DIR         = os.path.join(DATA_DIR, "dayend_balance/")
KYC_PATH        = os.path.join(DATA_DIR, "kyc.parquet")
TRAIN_PATH      = os.path.join(DATA_DIR, "train_labels.csv")
TEST_PATH       = os.path.join(DATA_DIR, "test.csv")
SAMPLE_SUB_PATH = os.path.join(DATA_DIR, "sample_submission.csv")
OUTPUT_PATH     = "predictions_new.csv"

N_SPLITS    = 5
SEED        = 42
CUTOFF      = pd.Timestamp("2024-04-01")
REF_DATE    = CUTOFF - pd.Timedelta(days=1)

SCALE_POS_WEIGHT = 519_565 / 75_435   # ≈ 6.888
TRX_TYPES = ["P2P", "MerchantPay", "BillPay", "CashIn", "CashOut"]

In [3]:
# ─────────────────────────────────────────────
# 1. INGEST
# ─────────────────────────────────────────────
print("Loading transactions via Dask...")
trx_ddf = dd.read_parquet(TRX_DIR + "trx_*.parquet")
trx_ddf["TRX_DATETIME"] = dd.to_datetime(trx_ddf["TRX_DATETIME"])
trx_ddf = trx_ddf[trx_ddf["TRX_DATETIME"] < CUTOFF]
trx_ddf["days_ago"] = (REF_DATE - trx_ddf["TRX_DATETIME"]).dt.days

print("Loading balances via Dask...")
bal_ddf = dd.read_parquet(BAL_DIR + "balance_*.parquet")
bal_ddf["DATE"] = dd.to_datetime(bal_ddf["DATE"])
bal_ddf = bal_ddf[bal_ddf["DATE"] < CUTOFF]

print("Loading KYC, labels, test IDs...")
kyc          = pd.read_parquet(KYC_PATH)
kyc["ACCOUNT_OPEN_DATE"] = pd.to_datetime(kyc["ACCOUNT_OPEN_DATE"])
train_labels = pd.read_csv(TRAIN_PATH)
test_ids     = pd.read_csv(TEST_PATH)

Loading transactions via Dask...
Loading balances via Dask...
Loading KYC, labels, test IDs...


In [4]:
# ─────────────────────────────────────────────
# 2. TRANSACTION WINDOW FEATURES
# ─────────────────────────────────────────────
print("Engineering transaction window features...")

def dask_window_agg(ddf, window_days, suffix):
    sub = ddf[ddf["days_ago"] <= window_days]
    grp = sub.groupby("SRC_ACCOUNT")["TRX_AMT"]
    feat = grp.agg(["count", "sum", "mean", "std", "max"]).compute().reset_index()
    feat.columns = [
        "ACCOUNT_ID",
        f"trx_count_{suffix}", f"trx_amt_sum_{suffix}", f"trx_amt_mean_{suffix}",
        f"trx_amt_std_{suffix}", f"trx_amt_max_{suffix}",
    ]
    feat[f"trx_amt_std_{suffix}"] = feat[f"trx_amt_std_{suffix}"].fillna(0)
    return feat

feat_7d  = dask_window_agg(trx_ddf, 7,  "7d")
feat_30d = dask_window_agg(trx_ddf, 30, "30d")
feat_90d = dask_window_agg(trx_ddf, 90, "90d")

# Fine-grained recent activity counts (1,2,3,5 days) — single combined groupby
print("Engineering fine-grained recent-activity counts...")
fine_ddf = trx_ddf[trx_ddf["days_ago"] <= 5].copy()
fine_counts_raw = (
    fine_ddf.groupby(["SRC_ACCOUNT", "days_ago"])
    .size()
    .compute()
    .reset_index(name="cnt")
    .pivot(index="SRC_ACCOUNT", columns="days_ago", values="cnt")
    .fillna(0)
    .reset_index()
    .rename(columns={"SRC_ACCOUNT": "ACCOUNT_ID"})
)
for d in range(6):
    if d not in fine_counts_raw.columns:
        fine_counts_raw[d] = 0
# Cumulative counts for 1d, 2d, 3d, 5d
for d in [1, 2, 3, 5]:
    cols = [c for c in range(0, d) if c in fine_counts_raw.columns]
    fine_counts_raw[f"trx_count_{d}d"] = fine_counts_raw[cols].sum(axis=1)
fine_counts = fine_counts_raw[["ACCOUNT_ID", "trx_count_1d", "trx_count_2d",
                                "trx_count_3d", "trx_count_5d"]]

# Overall recency (SRC side, any type)
recency = (
    trx_ddf.groupby("SRC_ACCOUNT")["days_ago"]
    .min().compute().reset_index()
    .rename(columns={"SRC_ACCOUNT": "ACCOUNT_ID", "days_ago": "days_since_last_trx"})
)

trx_90_ddf = trx_ddf[trx_ddf["days_ago"] <= 90]

# TRX_TYPE pivot
type_pivot_raw = (
    trx_90_ddf.groupby(["SRC_ACCOUNT", "TRX_TYPE"])
    .size().compute().unstack(fill_value=0).reset_index()
)
type_pivot_raw.index.name = None
type_pivot_raw.columns.name = None
for t in TRX_TYPES:
    if t not in type_pivot_raw.columns:
        type_pivot_raw[t] = 0
type_pivot = type_pivot_raw[["SRC_ACCOUNT"] + TRX_TYPES].rename(columns={"SRC_ACCOUNT": "ACCOUNT_ID"})
type_pivot.columns = ["ACCOUNT_ID" if c == "ACCOUNT_ID" else f"trxtype_{c}_90d" for c in type_pivot.columns]
total_90 = type_pivot.iloc[:, 1:].sum(axis=1).replace(0, 1)
for c in type_pivot.columns[1:]:
    type_pivot[c + "_ratio"] = type_pivot[c] / total_90

# Counterparty breadth — 90d and 30d
counterparty = (
    trx_90_ddf.groupby("SRC_ACCOUNT")["DST_ACCOUNT"]
    .nunique().compute().reset_index()
    .rename(columns={"SRC_ACCOUNT": "ACCOUNT_ID", "DST_ACCOUNT": "unique_counterparties_90d"})
)
trx_30_ddf = trx_ddf[trx_ddf["days_ago"] <= 30]
counterparty_30 = (
    trx_30_ddf.groupby("SRC_ACCOUNT")["DST_ACCOUNT"]
    .nunique().compute().reset_index()
    .rename(columns={"SRC_ACCOUNT": "ACCOUNT_ID", "DST_ACCOUNT": "unique_counterparties_30d"})
)

# Incoming P2P (90d)
p2p_recv = (
    trx_ddf[(trx_ddf["TRX_TYPE"] == "P2P") & (trx_ddf["days_ago"] <= 90)]
    .groupby("DST_ACCOUNT").size().compute().reset_index()
    .rename(columns={"DST_ACCOUNT": "ACCOUNT_ID", 0: "p2p_recv_count_90d"})
)


Engineering transaction window features...
Engineering fine-grained recent-activity counts...


In [5]:
# ─────────────────────────────────────────────
# 2b. PER-TYPE / PER-DIRECTION RECENCY
# ─────────────────────────────────────────────
print("Engineering per-type recency features...")

src_type_recency = (
    trx_ddf.groupby(["SRC_ACCOUNT", "TRX_TYPE"])["days_ago"]
    .min().compute().reset_index()
    .pivot(index="SRC_ACCOUNT", columns="TRX_TYPE", values="days_ago")
    .reset_index().rename(columns={"SRC_ACCOUNT": "ACCOUNT_ID"})
)
src_type_recency.columns = ["ACCOUNT_ID" if c == "ACCOUNT_ID" else f"days_since_last_{c}_sent"
                             for c in src_type_recency.columns]

dst_type_recency = (
    trx_ddf.groupby(["DST_ACCOUNT", "TRX_TYPE"])["days_ago"]
    .min().compute().reset_index()
    .pivot(index="DST_ACCOUNT", columns="TRX_TYPE", values="days_ago")
    .reset_index().rename(columns={"DST_ACCOUNT": "ACCOUNT_ID"})
)
dst_type_recency.columns = ["ACCOUNT_ID" if c == "ACCOUNT_ID" else f"days_since_last_{c}_recv"
                             for c in dst_type_recency.columns]

for t in TRX_TYPES:
    sc, dc = f"days_since_last_{t}_sent", f"days_since_last_{t}_recv"
    if sc not in src_type_recency.columns:
        src_type_recency[sc] = np.nan
    if dc not in dst_type_recency.columns:
        dst_type_recency[dc] = np.nan


Engineering per-type recency features...


In [6]:
# ─────────────────────────────────────────────
# 2c. WEEKLY-BINNED ACTIVITY
# ─────────────────────────────────────────────
print("Engineering weekly-binned activity features...")

trx_90_ddf2 = trx_90_ddf.copy()
trx_90_ddf2["week_bin"] = (trx_90_ddf2["days_ago"] // 7).astype("int8")

weekly = (
    trx_90_ddf2.groupby(["SRC_ACCOUNT", "week_bin"])["TRX_AMT"]
    .agg(["count", "sum"]).compute().reset_index()
)
weekly_count = weekly.pivot(index="SRC_ACCOUNT", columns="week_bin", values="count").fillna(0)
weekly_sum   = weekly.pivot(index="SRC_ACCOUNT", columns="week_bin", values="sum").fillna(0)
weekly_count.columns = [f"trx_count_w{int(c)}" for c in weekly_count.columns]
weekly_sum.columns   = [f"trx_sum_w{int(c)}" for c in weekly_sum.columns]
for w in range(13):
    cc, sc = f"trx_count_w{w}", f"trx_sum_w{w}"
    if cc not in weekly_count.columns:
        weekly_count[cc] = 0
    if sc not in weekly_sum.columns:
        weekly_sum[sc] = 0

weekly_feat = weekly_count.join(weekly_sum, how="outer").fillna(0).reset_index()
weekly_feat = weekly_feat.rename(columns={"SRC_ACCOUNT": "ACCOUNT_ID"})

weekly_feat["trx_count_7d_v2"]  = weekly_feat[[f"trx_count_w{w}" for w in range(1)]].sum(axis=1)
weekly_feat["trx_count_30d_v2"] = weekly_feat[[f"trx_count_w{w}" for w in range(4)]].sum(axis=1)
weekly_feat["trx_count_90d_v2"] = weekly_feat[[f"trx_count_w{w}" for w in range(13)]].sum(axis=1)
weekly_feat["trx_sum_7d_v2"]  = weekly_feat[[f"trx_sum_w{w}" for w in range(1)]].sum(axis=1)
weekly_feat["trx_sum_30d_v2"] = weekly_feat[[f"trx_sum_w{w}" for w in range(4)]].sum(axis=1)
weekly_feat["trx_sum_90d_v2"] = weekly_feat[[f"trx_sum_w{w}" for w in range(13)]].sum(axis=1)

week_cols  = [f"trx_count_w{w}" for w in range(13)]
weekly_arr = weekly_feat[week_cols].values

def _decay_slope(row):
    y = row[::-1]
    if y.sum() == 0:
        return 0.0
    x = np.arange(len(y))
    return stats.linregress(x, y).slope

weekly_feat["activity_decay_slope"] = np.apply_along_axis(_decay_slope, 1, weekly_arr)

def _last_active_week(row):
    nz = np.nonzero(row)[0]
    return int(nz[0]) if len(nz) > 0 else 13

weekly_feat["weeks_since_active"] = np.apply_along_axis(_last_active_week, 1, weekly_arr)


Engineering weekly-binned activity features...


In [7]:
# ─────────────────────────────────────────────
# 2d. BILL PAYMENT REGULARITY
# ─────────────────────────────────────────────
print("Engineering bill payment regularity features...")

bill_ddf = trx_90_ddf[trx_90_ddf["TRX_TYPE"] == "BillPay"].copy()
bill_ddf["month_bin"] = (bill_ddf["days_ago"] // 30).astype("int8")

bill_monthly = (
    bill_ddf.groupby(["SRC_ACCOUNT", "month_bin"]).size().compute()
    .reset_index(name="cnt")
    .pivot(index="SRC_ACCOUNT", columns="month_bin", values="cnt")
    .fillna(0).reset_index()
)
for m in range(3):
    if m not in bill_monthly.columns:
        bill_monthly[m] = 0
bill_monthly = bill_monthly.rename(columns={"SRC_ACCOUNT": "ACCOUNT_ID", 0: "bill_m0", 1: "bill_m1", 2: "bill_m2"})
bill_monthly = bill_monthly[["ACCOUNT_ID", "bill_m0", "bill_m1", "bill_m2"]]
bill_monthly["bill_missed_recent"] = ((bill_monthly["bill_m1"] > 0) & (bill_monthly["bill_m0"] == 0)).astype(np.int8)

bill_recency = (
    trx_ddf[trx_ddf["TRX_TYPE"] == "BillPay"]
    .groupby("SRC_ACCOUNT")["days_ago"].min().compute().reset_index()
    .rename(columns={"SRC_ACCOUNT": "ACCOUNT_ID", "days_ago": "days_since_last_billpay"})
)


Engineering bill payment regularity features...


In [8]:
# ─────────────────────────────────────────────
# 3. BALANCE FEATURES
# ─────────────────────────────────────────────
print("Engineering balance features...")

bal_stats = (
    bal_ddf.groupby("ACCOUNT_ID")["AVAILABLE_BALANCE"]
    .agg(["mean", "std", "min", "max", "first", "last"])
    .compute().reset_index()
)
bal_stats.columns = ["ACCOUNT_ID", "bal_mean", "bal_std", "bal_min", "bal_max", "bal_first", "bal_last"]
bal_stats["bal_std"] = bal_stats["bal_std"].fillna(0)
bal_stats["bal_cv"] = bal_stats["bal_std"] / (bal_stats["bal_mean"].abs() + 1e-9)
bal_stats["bal_drain_ratio"] = bal_stats["bal_last"] / (bal_stats["bal_first"].abs() + 1e-9)

bal_pd = bal_ddf[["ACCOUNT_ID", "DATE", "AVAILABLE_BALANCE"]].compute()
bal_pd = bal_pd.sort_values(["ACCOUNT_ID", "DATE"]).reset_index(drop=True)

def _slope(s):
    if len(s) < 2:
        return 0.0
    return stats.linregress(np.arange(len(s)), s.values).slope

bal_slope = (
    bal_pd.groupby("ACCOUNT_ID")["AVAILABLE_BALANCE"]
    .apply(_slope).reset_index().rename(columns={"AVAILABLE_BALANCE": "bal_slope"})
)

bal_last14_pd = bal_pd[bal_pd["DATE"] >= (CUTOFF - pd.Timedelta(days=14))]
bal_slope_14 = (
    bal_last14_pd.groupby("ACCOUNT_ID")["AVAILABLE_BALANCE"]
    .apply(_slope).reset_index().rename(columns={"AVAILABLE_BALANCE": "bal_slope_14d"})
)

zero_bal = (
    bal_pd.assign(is_zero=(bal_pd["AVAILABLE_BALANCE"] <= 0).astype(int))
    .groupby("ACCOUNT_ID")["is_zero"].sum().reset_index()
    .rename(columns={"is_zero": "zero_bal_days"})
)

bal_recent_pd = bal_pd[bal_pd["DATE"] >= (CUTOFF - pd.Timedelta(days=30))]
bal_early_pd  = bal_pd[bal_pd["DATE"] <  (CUTOFF - pd.Timedelta(days=60))]
bal_30d_avg = bal_recent_pd.groupby("ACCOUNT_ID")["AVAILABLE_BALANCE"].mean().reset_index().rename(columns={"AVAILABLE_BALANCE": "bal_30d_avg"})
bal_early_avg = bal_early_pd.groupby("ACCOUNT_ID")["AVAILABLE_BALANCE"].mean().reset_index().rename(columns={"AVAILABLE_BALANCE": "bal_early_avg"})

# ── Vectorized trailing-streak helper ──────────────────────────────────────
def trailing_streak_vectorized(df_sorted, cond_col, out_col):
    """df_sorted must be sorted by ACCOUNT_ID, DATE ascending."""
    d = df_sorted.copy()
    d["_not_cond"] = ~d[cond_col]
    d["_row_num"] = d.groupby("ACCOUNT_ID").cumcount()
    d["_break_row_num"] = np.where(d["_not_cond"], d["_row_num"], -1)
    d["_last_break"] = d.groupby("ACCOUNT_ID")["_break_row_num"].cummax()
    total_rows = d.groupby("ACCOUNT_ID")["_row_num"].transform("max") + 1
    last_rows = d.groupby("ACCOUNT_ID").tail(1).copy()
    last_rows[out_col] = last_rows["_row_num"] - last_rows["_last_break"]
    no_break_mask = last_rows["_last_break"] == -1
    last_rows.loc[no_break_mask, out_col] = total_rows.loc[last_rows.index][no_break_mask]
    return last_rows[["ACCOUNT_ID", out_col]].reset_index(drop=True)

print("Computing trailing balance streaks (vectorized)...")
bal_pd["_is_zero"] = bal_pd["AVAILABLE_BALANCE"] <= 0
trailing_zero_bal = trailing_streak_vectorized(bal_pd, "_is_zero", "trailing_zero_bal_days")

bal_pd["_acct_max"] = bal_pd.groupby("ACCOUNT_ID")["AVAILABLE_BALANCE"].transform("max")
bal_pd["_is_near_zero"] = bal_pd["AVAILABLE_BALANCE"] <= (0.01 * bal_pd["_acct_max"].clip(lower=1e-9))
trailing_near_zero_bal = trailing_streak_vectorized(bal_pd, "_is_near_zero", "trailing_near_zero_bal_days")

# Days since balance first dropped <= 10% of its own max
bal_pd["_is_drained_10pct"] = (bal_pd["_acct_max"] > 0) & (bal_pd["AVAILABLE_BALANCE"] <= 0.10 * bal_pd["_acct_max"])
drained_rows = bal_pd[bal_pd["_is_drained_10pct"]]
first_drain_date = (
    drained_rows.groupby("ACCOUNT_ID")["DATE"].min().reset_index()
    .rename(columns={"DATE": "first_drain_date"})
)
first_drain_date["days_since_drain"] = (CUTOFF - first_drain_date["first_drain_date"]).dt.days
drain_days = first_drain_date[["ACCOUNT_ID", "days_since_drain"]]


Engineering balance features...
Computing trailing balance streaks (vectorized)...


In [9]:
# ─────────────────────────────────────────────
# 4. KYC FEATURES
# ─────────────────────────────────────────────
print("Engineering KYC features...")
kyc_feat = kyc[kyc["ACCOUNT_TYPE"] == "Customer"].copy()
kyc_feat["account_age_days"] = (CUTOFF - kyc_feat["ACCOUNT_OPEN_DATE"]).dt.days

if "GENDER" in kyc_feat.columns:
    le = LabelEncoder()
    kyc_feat["GENDER_enc"] = le.fit_transform(kyc_feat["GENDER"].astype(str))
else:
    kyc_feat["GENDER_enc"] = -1

if "REGION" not in kyc_feat.columns:
    kyc_feat["REGION"] = "UNK"
kyc_feat["REGION"] = kyc_feat["REGION"].fillna("UNK")

kyc_feat = kyc_feat[["ACCOUNT_ID", "account_age_days", "GENDER_enc", "REGION"]]


Engineering KYC features...


In [10]:
# ─────────────────────────────────────────────
# 5. MERGE
# ─────────────────────────────────────────────
print("Merging features...")

FEAT_FRAMES = [
    feat_7d, feat_30d, feat_90d, fine_counts,
    recency, type_pivot, counterparty, counterparty_30, p2p_recv,
    src_type_recency, dst_type_recency,
    weekly_feat,
    bill_monthly, bill_recency,
    bal_stats, bal_slope, bal_slope_14, drain_days, zero_bal, bal_30d_avg, bal_early_avg,
    trailing_zero_bal, trailing_near_zero_bal,
    kyc_feat,
]

def merge_all(account_ids):
    base = pd.DataFrame({"ACCOUNT_ID": account_ids})
    for df in FEAT_FRAMES:
        base = base.merge(df, on="ACCOUNT_ID", how="left")
    return base

train_feat = merge_all(train_labels["ACCOUNT_ID"])
test_feat  = merge_all(test_ids["ACCOUNT_ID"])

# Sentinels for "never drained / never near-zero" — distinct from "no data"
for df in [train_feat, test_feat]:
    df["days_since_drain"] = df["days_since_drain"].fillna(-1)


Merging features...


In [11]:
# ─────────────────────────────────────────────
# 5b. POST-MERGE INTERACTION FEATURES
# ─────────────────────────────────────────────
print("Engineering interaction features...")

for df in [train_feat, test_feat]:
    df["activity_momentum"] = df["trx_count_30d"] / (df["days_since_last_trx"].fillna(999) + 1)

    df["channel_shift_ratio"] = (
        df["trx_count_30d"] / ((df["trx_count_90d"] / 3.0).replace(0, np.nan))
    )

    df["is_silent_30d"] = (df["trx_count_30d"].fillna(0) == 0).astype(np.int8)
    df["is_silent_7d"]  = (df["trx_count_7d"].fillna(0)  == 0).astype(np.int8)
    df["is_silent_3d"]  = (df["trx_count_3d"].fillna(0)  == 0).astype(np.int8)

    df["bal_recent_vs_early"] = df["bal_30d_avg"] / (df["bal_early_avg"].abs() + 1e-9)

    df["p2p_social_ratio"] = (
        df.get("trxtype_P2P_90d", pd.Series(0, index=df.index))
        / (df["p2p_recv_count_90d"].fillna(0) + 1)
    )

    df["sum_shift_ratio_7_30"] = (
        df["trx_amt_sum_7d"] / ((df["trx_amt_sum_30d"] / 4.0).replace(0, np.nan))
    )
    df["sum_shift_ratio_30_90"] = (
        df["trx_amt_sum_30d"] / ((df["trx_amt_sum_90d"] / 3.0).replace(0, np.nan))
    )
    df["mean_amt_shift_30_90"] = (
        df["trx_amt_mean_30d"] / (df["trx_amt_mean_90d"].replace(0, np.nan))
    )
    df["counterparty_shift_30_90"] = (
        df["unique_counterparties_30d"] / (df["unique_counterparties_90d"].replace(0, np.nan) / 3.0)
    )

    df["activity_per_tenure"] = df["trx_count_90d"] / (df["account_age_days"].replace(0, np.nan) + 1)

    df["bal_last_vs_30davg"] = df["bal_last"] / (df["bal_30d_avg"].abs() + 1e-9)

    # Combined dormancy signals
    df["dormant_combo"] = (
        df["trailing_near_zero_bal_days"].fillna(0) * (df["trx_count_3d"].fillna(0) == 0).astype(int)
    )
    df["is_fully_dormant"] = (
        (df["trx_count_3d"].fillna(0) == 0) &
        (df["trailing_near_zero_bal_days"].fillna(0) > 0)
    ).astype(np.int8)
    df["is_fully_dormant_30d"] = (
        (df["trx_count_30d"].fillna(0) == 0) &
        (df["trailing_zero_bal_days"].fillna(0) > 0)
    ).astype(np.int8)

train_feat = train_feat.merge(train_labels[["ACCOUNT_ID", "CHURN"]], on="ACCOUNT_ID")


Engineering interaction features...


In [12]:
# ─────────────────────────────────────────────
# 6. MATRICES — split ratio/slope cols (median-impute + isnan flag) vs count cols (-999)
# ─────────────────────────────────────────────
print("Building matrices...")

for df in [train_feat, test_feat]:
    df.replace([np.inf, -np.inf], np.nan, inplace=True)

DROP_COLS = ["ACCOUNT_ID", "CHURN", "REGION"]
BASE_FEAT_COLS = [c for c in train_feat.columns if c not in DROP_COLS]

# Ratio/slope/momentum-type columns -> median impute + isnan flag
RATIO_SLOPE_COLS = [c for c in BASE_FEAT_COLS if any(
    k in c for k in ["ratio", "slope", "momentum", "_cv", "shift", "_vs_"]
)]
COUNT_LIKE_COLS = [c for c in BASE_FEAT_COLS if c not in RATIO_SLOPE_COLS]

for df in [train_feat, test_feat]:
    for c in RATIO_SLOPE_COLS:
        df[c + "_isnan"] = df[c].isna().astype(np.int8)

ISNAN_COLS = [c + "_isnan" for c in RATIO_SLOPE_COLS]
FEAT_COLS = BASE_FEAT_COLS + ISNAN_COLS

# Median impute ratio/slope cols (median computed on TRAIN only, applied to both)
for c in RATIO_SLOPE_COLS:
    med = train_feat[c].median()
    if pd.isna(med):
        med = 0.0
    train_feat[c] = train_feat[c].fillna(med)
    test_feat[c]  = test_feat[c].fillna(med)

# -999 sentinel for count-like columns (missing = no activity)
for c in COUNT_LIKE_COLS:
    train_feat[c] = train_feat[c].fillna(-999.0)
    test_feat[c]  = test_feat[c].fillna(-999.0)

X      = train_feat[FEAT_COLS].values.astype(np.float32)
y      = train_feat["CHURN"].values.astype(np.int8)
X_test = test_feat[FEAT_COLS].values.astype(np.float32)

region_train = train_feat["REGION"].fillna("UNK").values
region_test  = test_feat["REGION"].fillna("UNK").values

print(f"Train shape : {X.shape}  |  Churn rate: {y.mean():.3%}")
print(f"Test  shape : {X_test.shape}")
print(f"  Ratio/slope cols (median-imputed): {len(RATIO_SLOPE_COLS)}")
print(f"  Count-like cols (-999 sentinel)  : {len(COUNT_LIKE_COLS)}")


Building matrices...
Train shape : (595000, 137)  |  Churn rate: 12.678%
Test  shape : (255000, 137)
  Ratio/slope cols (median-imputed): 19
  Count-like cols (-999 sentinel)  : 99


In [13]:
# ─────────────────────────────────────────────
# 7. LIGHTGBM CV  (OOF REGION target encoding inside each fold)
# ─────────────────────────────────────────────
print("\nTraining LightGBM with StratifiedKFold CV...")

LGB_PARAMS = {
    "objective":         "binary",
    "metric":            ["auc", "average_precision"],
    "boosting_type":     "gbdt",
    "num_leaves":         255,
    "max_depth":          -1,
    "learning_rate":      0.02,
    "n_estimators":       6000,
    "feature_fraction":   0.8,
    "bagging_fraction":   0.8,
    "bagging_freq":       5,
    "min_child_samples":  30,
    "reg_alpha":          0.1,
    "reg_lambda":         1.0,
    "scale_pos_weight":   SCALE_POS_WEIGHT,
    "random_state":       SEED,
    "n_jobs":            -1,
    "verbose":           -1,
}

skf        = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
oof_lgb    = np.zeros(len(X), dtype=np.float64)
test_lgb   = np.zeros(len(X_test), dtype=np.float64)
oof_cb     = np.zeros(len(X), dtype=np.float64)
test_cb    = np.zeros(len(X_test), dtype=np.float64)
fold_aucs_lgb, fold_aucs_cb = [], []

REGION_GLOBAL_MEAN = y.mean()
FULL_FEAT_COLS = FEAT_COLS + ["REGION_target_enc"]

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_tr, X_val = X[tr_idx], X[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]

    # OOF REGION target encoding
    tr_region_df = pd.DataFrame({"REGION": region_train[tr_idx], "CHURN": y_tr})
    region_means = tr_region_df.groupby("REGION")["CHURN"].mean()

    tr_region_enc   = pd.Series(region_train[tr_idx]).map(region_means).fillna(REGION_GLOBAL_MEAN).values
    val_region_enc  = pd.Series(region_train[val_idx]).map(region_means).fillna(REGION_GLOBAL_MEAN).values
    test_region_enc = pd.Series(region_test).map(region_means).fillna(REGION_GLOBAL_MEAN).values

    X_tr_full   = np.hstack([X_tr,  tr_region_enc.reshape(-1, 1)]).astype(np.float32)
    X_val_full  = np.hstack([X_val, val_region_enc.reshape(-1, 1)]).astype(np.float32)
    X_test_full = np.hstack([X_test, test_region_enc.reshape(-1, 1)]).astype(np.float32)

    # ---- LightGBM ----
    lgb_model = lgb.LGBMClassifier(**LGB_PARAMS)
    lgb_model.fit(
        X_tr_full, y_tr,
        eval_set=[(X_val_full, y_val)],
        eval_metric="auc",
        callbacks=[lgb.early_stopping(stopping_rounds=200, verbose=False),
                   lgb.log_evaluation(period=-1)],
    )
    val_prob_lgb = lgb_model.predict_proba(X_val_full)[:, 1]
    oof_lgb[val_idx] = val_prob_lgb
    test_lgb += lgb_model.predict_proba(X_test_full)[:, 1] / N_SPLITS
    auc_lgb = roc_auc_score(y_val, val_prob_lgb)
    fold_aucs_lgb.append(auc_lgb)

    # ---- CatBoost ----
    cb_model = CatBoostClassifier(
        iterations=4000,
        learning_rate=0.03,
        depth=8,
        l2_leaf_reg=3.0,
        scale_pos_weight=SCALE_POS_WEIGHT,
        eval_metric="AUC",
        random_seed=SEED,
        early_stopping_rounds=200,
        verbose=False,
    )
    cb_model.fit(X_tr_full, y_tr, eval_set=(X_val_full, y_val))
    val_prob_cb = cb_model.predict_proba(X_val_full)[:, 1]
    oof_cb[val_idx] = val_prob_cb
    test_cb += cb_model.predict_proba(X_test_full)[:, 1] / N_SPLITS
    auc_cb = roc_auc_score(y_val, val_prob_cb)
    fold_aucs_cb.append(auc_cb)

    print(f"  Fold {fold}: LGB AUC={auc_lgb:.5f}  best_iter={lgb_model.best_iteration_}  "
          f"| CB AUC={auc_cb:.5f}  best_iter={cb_model.best_iteration_}")

    if fold == 1:
        importances = pd.Series(lgb_model.feature_importances_, index=FULL_FEAT_COLS)
        print("\n  Top 20 LGB features (fold 1 gain):")
        print(importances.sort_values(ascending=False).head(20).to_string())
        print()

print(f"\nLGB OOF AUC : {roc_auc_score(y, oof_lgb):.5f}  (±{np.std(fold_aucs_lgb):.5f})")
print(f"CB  OOF AUC : {roc_auc_score(y, oof_cb):.5f}  (±{np.std(fold_aucs_cb):.5f})")



Training LightGBM with StratifiedKFold CV...
  Fold 1: LGB AUC=0.98424  best_iter=244  | CB AUC=0.98423  best_iter=478

  Top 20 LGB features (fold 1 gain):
bal_slope_14d           1934
activity_decay_slope    1346
activity_momentum       1278
account_age_days        1249
trx_count_90d           1184
trx_sum_w5              1110
trx_sum_w8              1103
trx_sum_w9              1080
trx_sum_w1              1073
trx_sum_w11             1054
channel_shift_ratio     1052
trx_sum_w7              1035
trx_sum_w3              1032
activity_per_tenure     1029
trx_sum_w2              1028
trx_sum_w4              1014
days_since_last_trx      998
trx_sum_w6               997
trx_sum_w10              957
bal_last_vs_30davg       956

  Fold 2: LGB AUC=0.98435  best_iter=183  | CB AUC=0.98437  best_iter=237
  Fold 3: LGB AUC=0.98502  best_iter=215  | CB AUC=0.98496  best_iter=385
  Fold 4: LGB AUC=0.98460  best_iter=199  | CB AUC=0.98464  best_iter=601
  Fold 5: LGB AUC=0.98456  best_iter=25

In [14]:
# ─────────────────────────────────────────────
# EXPLAINABILITY ARTIFACTS
# ─────────────────────────────────────────────
import shap
import matplotlib.pyplot as plt
import os

os.makedirs("explainability", exist_ok=True)

# Use fold-1's trained LightGBM model (already fitted with REGION_target_enc appended)
# Reconstruct fold-1's training matrix with region encoding for consistency
tr_idx_f1, val_idx_f1 = list(skf.split(X, y))[0]
tr_region_df_f1 = pd.DataFrame({"REGION": region_train[tr_idx_f1], "CHURN": y[tr_idx_f1]})
region_means_f1 = tr_region_df_f1.groupby("REGION")["CHURN"].mean()
train_region_enc_f1 = pd.Series(region_train).map(region_means_f1).fillna(REGION_GLOBAL_MEAN).values
X_full = np.hstack([X, train_region_enc_f1.reshape(-1, 1)]).astype(np.float32)

# Sample for SHAP speed
rng = np.random.RandomState(SEED)
sample_idx = rng.choice(len(X_full), size=5000, replace=False)
X_sample = X_full[sample_idx]

explainer = shap.TreeExplainer(lgb_model)  # lgb_model = fold-1 model from the CV loop
shap_values = explainer.shap_values(X_sample)
if isinstance(shap_values, list):  # binary classifier sometimes returns [class0, class1]
    shap_values = shap_values[1]

# 1. Summary (beeswarm) plot
shap.summary_plot(shap_values, X_sample, feature_names=FULL_FEAT_COLS, show=False)
plt.tight_layout()
plt.savefig("explainability/shap_summary.png", dpi=150, bbox_inches="tight")
plt.close()

# 2. Mean |SHAP| bar plot
shap.summary_plot(shap_values, X_sample, feature_names=FULL_FEAT_COLS,
                  plot_type="bar", show=False)
plt.tight_layout()
plt.savefig("explainability/shap_importance_bar.png", dpi=150, bbox_inches="tight")
plt.close()

# 3. Dependence plot for the top feature
mean_abs_shap = np.abs(shap_values).mean(axis=0)
top_idx = int(np.argmax(mean_abs_shap))
shap.dependence_plot(top_idx, shap_values, X_sample, feature_names=FULL_FEAT_COLS, show=False)
plt.tight_layout()
plt.savefig("explainability/shap_dependence_top.png", dpi=150, bbox_inches="tight")
plt.close()

# Print top-15 SHAP features for the insights writeup
top15_idx = np.argsort(mean_abs_shap)[::-1][:15]
print("Top 15 features by mean |SHAP|:")
for i in top15_idx:
    print(f"  {FULL_FEAT_COLS[i]:35s}  {mean_abs_shap[i]:.5f}")

Top 15 features by mean |SHAP|:
  trx_count_7d                         0.94497
  trx_count_30d_v2                     0.75854
  days_since_last_trx                  0.46227
  trx_count_30d                        0.37638
  trx_count_5d                         0.22828
  trx_count_90d                        0.18926
  trx_count_90d_v2                     0.15017
  activity_momentum                    0.09181
  trx_count_w1                         0.09028
  trx_sum_w-1                          0.07267
  bal_slope_14d                        0.06304
  trx_count_w0                         0.06126
  weeks_since_active                   0.06006
  trx_sum_w9                           0.05202
  trx_amt_std_7d                       0.04929


In [15]:
# ─────────────────────────────────────────────
# 7b. BLEND SEARCH
# ─────────────────────────────────────────────
print("\nSearching blend weight (LGB vs CatBoost)...")
best_w, best_auc = 0.5, 0.0
for w in np.arange(0.0, 1.01, 0.05):
    blend = w * oof_lgb + (1 - w) * oof_cb
    auc = roc_auc_score(y, blend)
    if auc > best_auc:
        best_auc, best_w = auc, w

print(f"Best blend weight (LGB): {best_w:.2f}  ->  OOF AUC = {best_auc:.5f}")

oof_blend  = best_w * oof_lgb + (1 - best_w) * oof_cb
test_blend = best_w * test_lgb + (1 - best_w) * test_cb

oof_ap = average_precision_score(y, oof_blend)
print(f"Blended OOF AUC : {roc_auc_score(y, oof_blend):.5f}")
print(f"Blended OOF AP  : {oof_ap:.5f}")

k = int(np.ceil(len(oof_blend) * 0.10))
top10_thresh = np.sort(oof_blend)[::-1][k - 1]
top10_mask = oof_blend >= top10_thresh
print(f"Precision@10% : {y[top10_mask].mean():.5f}")
print(f"Recall@10%    : {y[top10_mask].sum() / max(y.sum(), 1):.5f}")


Searching blend weight (LGB vs CatBoost)...
Best blend weight (LGB): 0.45  ->  OOF AUC = 0.98460
Blended OOF AUC : 0.98460
Blended OOF AP  : 0.91982
Precision@10% : 0.90975
Recall@10%    : 0.71757


In [16]:
# ─────────────────────────────────────────────
# 8. SUBMISSION
# ─────────────────────────────────────────────
print("\nGenerating submission...")
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)

test_probs = np.clip(test_blend, 0.0, 1.0)

submission = pd.DataFrame({
    "ACCOUNT_ID": test_ids["ACCOUNT_ID"],
    "CHURN_PROB": test_probs,
})
submission = sample_sub[["ACCOUNT_ID"]].merge(submission, on="ACCOUNT_ID", how="left")
submission["CHURN_PROB"] = submission["CHURN_PROB"].fillna(0.0).round(6)

submission.to_csv(OUTPUT_PATH, index=False)
print(f"Saved → {OUTPUT_PATH}  ({len(submission):,} rows)")
print(f"Predicted mean churn prob : {submission['CHURN_PROB'].mean():.4f}")
print(f"Predicted >0.5 rate       : {(submission['CHURN_PROB'] > 0.5).mean():.3%}")


Generating submission...
Saved → predictions_new.csv  (255,000 rows)
Predicted mean churn prob : 0.1884
Predicted >0.5 rate       : 20.252%
